# Lesson 21 | What changes when the network becomes larger?

A design that looks balanced on a tiny graph can become limited by memory, queues, or a small set of high-traffic neurons at larger scale.

Today asks one question:

> **How do we identify which stage is limiting the workload at a particular scale?**

Primary new concept: **bottleneck as the stage with the highest utilization relative to capacity.**

## 1. Concept ledger

**Already known:** latency, throughput, bandwidth, **First-In First-Out (FIFO)** backpressure, external **Double Data Rate (DDR)** memory, and sparse fan-out.

**One primary concept today:** **utilization**, used to identify a scale-dependent **bottleneck**.

**Supporting term:** a **hotspot** only means traffic/work is concentrated on a small set of resources. This lesson only asks you to recognize that hotspots can change demand distribution; bank conflicts and cache optimization are not mastery targets yet.

**Preview only:** measured 10K/50K MaleCNS runs, bank conflicts, telemetry, and optimization such as caching or multiple synapse engines.

## 2. Bottleneck is a relationship, not a permanent label

For one stage, use the minimal definition:

`utilization = demand / capacity`

A stage near or above 1.0 has little or no headroom.

The bottleneck can move when the graph, event rate, memory pattern, or architecture changes. “DDR is always the bottleneck” is not a specification.

## 3. Pipeline picture

<div style="max-width:860px; margin:1rem auto;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 860 220" role="img" aria-label="event pipeline stages" style="width:100%; height:auto; display:block;">
  <defs>
    <marker id="l21-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#2f5f3f"/>
    </marker>
  </defs>
  <g font-family="sans-serif" font-size="20" text-anchor="middle">
    <rect x="20" y="75" width="145" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="92" y="117" fill="#1f2d24">spike source</text>
    <rect x="190" y="75" width="115" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="247" y="117" fill="#1f2d24">FIFO</text>
    <rect x="330" y="75" width="165" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="412" y="117" fill="#1f2d24">synapse lookup</text>
    <rect x="520" y="75" width="145" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="592" y="105" fill="#1f2d24">DDR /</text><text x="592" y="130" fill="#1f2d24">storage</text>
    <rect x="690" y="75" width="150" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="765" y="117" fill="#1f2d24">target update</text>
  </g>
  <g fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l21-arrow)">
    <path d="M165 110 L190 110"/><path d="M305 110 L330 110"/><path d="M495 110 L520 110"/><path d="M665 110 L690 110"/>
  </g>
</svg>
</div>

Every stage can have a different capacity and demand.

## 4. Run: synthetic scale study

These are **teaching numbers**, not measured FPGA results. They exist only to practice the analysis method.

In [ ]:
capacities = {
    "fifo": 10.0,
    "lookup": 8.0,
    "memory": 6.0,
    "update": 9.0,
}

cases = {
    "small": {"fifo": 2.0, "lookup": 3.6, "memory": 1.8, "update": 2.2},
    "medium": {"fifo": 5.0, "lookup": 5.5, "memory": 5.7, "update": 4.8},
    "large": {"fifo": 9.5, "lookup": 6.5, "memory": 5.4, "update": 6.8},
}

for name, demand in cases.items():
    utilization = {
        stage: demand[stage] / capacities[stage]
        for stage in capacities
    }
    bottleneck = max(utilization, key=utilization.get)
    print(name, "bottleneck:", bottleneck,
          "utilization:", round(utilization[bottleneck], 2))

## 5. Observe

The highest utilization is lookup in the small case, memory in the medium case, and FIFO in the large case.

That is the point of the lesson: a bottleneck is determined by **demand relative to capacity** and can move as workload scale and traffic distribution change. This toy diagnosis says where to investigate first; it does not automatically prove the deeper cause.

## 6. Hotspots are uneven work

Two networks can have the same total number of edges but very different traffic distributions. A few high-fanout or high-rate sources can create queue pressure or bank conflicts.

So “network size” is not a complete performance description. Distribution matters.

## 7. Measure before optimizing

RMD-022 intentionally comes after the correctness baseline.

Caching, banking, lazy updates, or more engines are not automatically improvements. First establish:

1. a correct baseline;
2. telemetry and workload definition;
3. the observed limiting resource;
4. a before/after benchmark with the same workload.

## 8. Try It

In the medium case, increase only memory capacity. Predict where the bottleneck moves.

Then in the large case, increase only FIFO capacity. Which stage now has the highest utilization?

## 9. Exercise

[Lesson 21 exercise: identify the highest-utilization stage](../../exercises/en/21_scaling_bottlenecks.ipynb)

## 10. AI Task

Give an AI a utilization table and ask for three possible causes of the hottest stage. Require it to label them as hypotheses, not measurements.

## 11. Human Check

Explain why the bottleneck can move with scale. Why is “largest raw workload” not necessarily the same as “highest utilization”? What evidence is required before choosing an optimization?

## 12. Engineering Handoff

Maps to `RMD-019~022`, `MOD-014 telemetry`, and performance metrics `P-001~P-008`. Formal claims require measured reports on a defined workload; the numbers in this lesson are only teaching fixtures.

## 13. Project Trace

- Lesson: `LSN-021`
- Mapping: `RMD-019 / RMD-020 / RMD-021 / RMD-022`
- Requirement paths: `TRACE-P-001`, `TRACE-F-001`
- Correctness oracle before optimization: `T-016`
- Performance metrics: `P-001~P-008`

## 14. Exit Ticket

Given demand and capacity for several stages, you can compute utilization, identify the current bottleneck candidate, and explain why that diagnosis may change at another scale.